In [ ]:
import pandas as pd
import numpy as np

# 1. Cargar el dataset generado anteriormente
input_path = '/content/cleaned_routes.csv'
df = pd.read_csv(input_path)
original_rows = len(df)

# 2. Renombrar columnas a formato simple para C++
column_map = {
    'Airline': 'airline',
    'Airline ID': 'airline_id',
    'Source Airport': 'source_iata',
    'Source Airport ID': 'source_id',
    'Destination Airport': 'destination_iata',
    'Destination Airport ID': 'destination_id',
    'Codeshare': 'codeshare',
    'Stops': 'stops',
    'Equipment': 'equipment'
}
df.rename(columns=column_map, inplace=True)

# 3. Reemplazar valores nulos
df.replace([r'\\N', r'\N', '', 'nan', 'NaN'], np.nan, inplace=True)

# 4. Eliminar filas donde falte source_id o destination_id
initial_valid_ids = len(df)
df.dropna(subset=['source_id', 'destination_id'], inplace=True)
rows_removed_missing_ids = initial_valid_ids - len(df)

# 5. Convertir a tipos de datos correctos
df['source_id'] = df['source_id'].astype(int)
df['destination_id'] = df['destination_id'].astype(int)
df['stops'] = pd.to_numeric(df['stops'], errors='coerce').fillna(0).astype(int)

# 6. Eliminar rutas circulares (self-loops)
initial_no_loops = len(df)
df = df[df['source_id'] != df['destination_id']]
self_loops_removed = initial_no_loops - len(df)

# 7. Validar códigos IATA
def is_valid_iata(code):
    if pd.isna(code): return False
    return len(str(code)) == 3

df = df[df['source_iata'].apply(is_valid_iata) & df['destination_iata'].apply(is_valid_iata)]

# --- GENERACIÓN DE ARCHIVOS ESPECÍFICOS ---

# A. routes_graph_clean.csv
routes_graph = df[['source_id', 'destination_id', 'source_iata', 'destination_iata']].copy()
initial_graph_len = len(routes_graph)
routes_graph.drop_duplicates(subset=['source_id', 'destination_id'], inplace=True)
duplicates_graph = initial_graph_len - len(routes_graph)
routes_graph.to_csv('routes_graph_clean.csv', index=False)

# B. routes_airline_clean.csv
routes_airline = df[['airline', 'source_id', 'destination_id', 'source_iata', 'destination_iata']].copy()
routes_airline.dropna(subset=['airline'], inplace=True)
initial_airline_len = len(routes_airline)
routes_airline.drop_duplicates(subset=['airline', 'source_id', 'destination_id'], inplace=True)
duplicates_airline = initial_airline_len - len(routes_airline)
routes_airline.to_csv('routes_airline_clean.csv', index=False)

# C. routes_full_clean.csv
routes_full = df.copy()
routes_full.to_csv('routes_full_clean.csv', index=False)


=== REPORTE DE LIMPIEZA DE RUTAS ===
Filas originales: 67663
Filas eliminadas por IDs faltantes: 423
Self-loops (origen == destino) eliminados: 1
Duplicados eliminados (ID Origen + ID Destino): 29966
Duplicados eliminados (Aerolínea + IDs): 0
------------------------------------
Filas finales en routes_graph_clean.csv: 37273
Filas finales en routes_airline_clean.csv: 67239
Filas finales en routes_full_clean.csv: 67239
------------------------------------
Aerolíneas únicas: 568
Aeropuertos origen únicos: 3315
Aeropuertos destino únicos: 3321


In [ ]:
import pandas as pd
import numpy as np

# 1. Cargar el dataset de aeropuertos limpiado inicialmente
input_path = '/content/cleaned_airports.csv'
df_air = pd.read_csv(input_path)
original_count = len(df_air)

# 2. Renombrar columnas a formato simple para C++
air_column_map = {
    'Airport ID': 'id', 'Name': 'name', 'City': 'city', 'Country': 'country',
    'IATA': 'iata', 'ICAO': 'icao', 'Latitude': 'latitude', 'Longitude': 'longitude',
    'Altitude': 'altitude', 'Timezone_Offset': 'timezone', 'DST': 'dst',
    'Tz_Database_Time_Zone': 'tz', 'Type': 'type', 'Source': 'source'
}
df_air.rename(columns=air_column_map, inplace=True)

# 3. Reemplazar valores nulos
df_air.replace([r'\\N', r'\N', '', 'nan', 'NaN', 'None'], np.nan, inplace=True)

# 4. Limpieza de textos
for col in df_air.select_dtypes(include=['object']).columns:
    df_air[col] = df_air[col].astype(str).str.strip().replace('nan', np.nan)

# 5. Validar códigos IATA (3 letras) e ICAO (4 letras)
iata_invalid_count = 0
icoa_invalid_count = 0

def clean_iata(val):
    global iata_invalid_count
    if pd.isna(val) or val == 'nan': return np.nan
    if len(str(val)) == 3: return str(val)
    iata_invalid_count += 1
    return np.nan

def clean_icao(val):
    global icoa_invalid_count
    if pd.isna(val) or val == 'nan': return np.nan
    if len(str(val)) == 4: return str(val)
    icoa_invalid_count += 1
    return np.nan

df_air['iata'] = df_air['iata'].apply(clean_iata)
df_air['icao'] = df_air['icao'].apply(clean_icao)

# 6. Eliminar filas con campos críticos faltantes
initial_rows = len(df_air)
df_air.dropna(subset=['id', 'name', 'country', 'latitude', 'longitude'], inplace=True)
critical_removed = initial_rows - len(df_air)

# 7. Convertir tipos y validar coordenadas
df_air['id'] = df_air['id'].astype(int)
df_air['latitude'] = pd.to_numeric(df_air['latitude'], errors='coerce')
df_air['longitude'] = pd.to_numeric(df_air['longitude'], errors='coerce')

initial_coords = len(df_air)
df_air = df_air[
    (df_air['latitude'] >= -90) & (df_air['latitude'] <= 90) &
    (df_air['longitude'] >= -180) & (df_air['longitude'] <= 180)
]
coords_removed = initial_coords - len(df_air)

# 8. Verificar IDs únicos
initial_ids = len(df_air)
df_air.drop_duplicates(subset=['id'], keep='first', inplace=True)
duplicates_removed = initial_ids - len(df_air)

# 9. Crear lista de Sudamérica
south_america_countries = [
    'Argentina', 'Bolivia', 'Brazil', 'Chile', 'Colombia', 'Ecuador',
    'Guyana', 'Paraguay', 'Peru', 'Suriname', 'Uruguay', 'Venezuela', 'French Guiana'
]
df_south_america = df_air[df_air['country'].isin(south_america_countries)].copy()

# 10. Generar archivos finales
columns_final = ['id', 'name', 'city', 'country', 'iata', 'icao', 'latitude', 'longitude']

df_air[columns_final].to_csv('airports_clean_final.csv', index=False)
df_air.to_csv('airports_full_clean.csv', index=False)
df_south_america[columns_final].to_csv('airports_south_america_clean.csv', index=False)



=== REPORTE DE LIMPIEZA DE AEROPUERTOS ===
Filas originales: 7698
Eliminados por campos críticos faltantes: 0
Eliminados por coordenadas inválidas: 0
IDs duplicados eliminados: 0
IATA inválidos corregidos a nulo: 0
ICAO inválidos corregidos a nulo: 5
------------------------------------------
Total aeropuertos finales: 7698
Aeropuertos con IATA: 6072
Aeropuertos sin IATA: 1626
Países únicos: 237
Aeropuertos en Sudamérica: 703
